# 7 · Multimodal features

Multimodal models score from precomputed item vectors (an image encoder's
output, a text embedding) beside the interactions. This guide gives
MovieLens-100K two such modalities and covers:

- producing a feature matrix for the items, here a text embedding of the
  title and genres;
- the three file layouts `reader.multimodal` reads (a numpy array with its
  row-order file, a tabular file, a parquet file) and why the array needs the
  row-order file;
- what happens to items a feature file does not cover, and `normalize`;
- the `MultiModalFeatures` object the dataset carries, and what models see
  of it through `info()`;
- choosing modalities per model, and a short VBPR run next to BPR.

**Requirements:** `pip install warprec jupyter`. Runs in one to two minutes on
a laptop CPU; it downloads MovieLens-100K (5 MB) into `guides/data/`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from loguru import logger

sys.path.insert(0, "..")
from guide_data import movielens_100k

folder = movielens_100k()
RUNS = Path("../runs/07-multimodal-features")
RUNS.mkdir(parents=True, exist_ok=True)
logger.disable("warprec")

`movielens_100k()` downloads the data and writes `ratings.tsv` and `items.tsv`
as guide 1 showed. The item file has a title and the genres of every film:

In [2]:
items = pd.read_csv(folder / "items.tsv", sep="\t")
items.head(3)

,item_id,title,release_date,genres
0,1,Toy Story (1995),01-Jan-1995,Animation|Children's|Comedy
1,2,GoldenEye (1995),01-Jan-1995,Action|Adventure|Thriller
2,3,Four Rooms (1995),01-Jan-1995,Thriller


## Producing the vectors

WarpRec does not extract features: it reads vectors that were computed
elsewhere, one per item. A real project would put the poster through an image
encoder or the plot through a pretrained sentence encoder. To keep this guide
free of extra dependencies, the text modality is a TF-IDF of the title and
genres reduced to 32 dimensions with a truncated SVD, both from scikit-learn
(already a WarpRec dependency). Any encoder's output is used the same way.

In [3]:
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

documents = (
    items.title.str.replace(r"\s*\(\d{4}\)$", "", regex=True)
    + " "
    + items.genres.str.replace("|", " ")
)
tfidf = TfidfVectorizer(min_df=2, sublinear_tf=True).fit_transform(documents)
text = TruncatedSVD(n_components=32, random_state=42).fit_transform(tfidf)
text = text.astype("float32")
print(documents[0], "->", text.shape)

Toy Story Animation Children's Comedy -> (1682, 32)


A quick check that the vectors mean something: the nearest titles of a few
films by cosine similarity.

In [4]:
def nearest(matrix: np.ndarray, row_of: dict, title: str, n: int = 4) -> list:
    """The n titles whose rows of `matrix` are closest to the row of `title`."""
    unit = matrix / np.maximum(np.linalg.norm(matrix, axis=1, keepdims=True), 1e-9)
    ids = list(row_of)
    scores = unit[[row_of[i] for i in ids]] @ unit[row_of[item_of[title]]]
    return [title_of[ids[i]] for i in np.argsort(-scores)[1 : n + 1]]


title_of = dict(zip(items.item_id, items.title))
item_of = dict(zip(items.title, items.item_id))
row_of_item = {item: row for row, item in enumerate(items.item_id)}
for title in ["Star Wars (1977)", "Toy Story (1995)"]:
    print(title, "->", nearest(text, row_of_item, title))

Star Wars (1977) -> ['Empire Strikes Back, The (1980)', 'Starship Troopers (1997)', 'Return of the Jedi (1983)', 'Star Trek: Generations (1994)']
Toy Story (1995) -> ['Balto (1995)', 'Pinocchio (1940)', 'Oliver & Company (1988)', 'Goofy Movie, A (1995)']


The second modality is the genre flags as a 19-column one-hot matrix. It is
the same information side information carries (guide 4), used here only to
show that `reader.multimodal` holds any number of named modalities.

In [5]:
genres = items.genres.str.get_dummies("|").astype("float32")
genres.shape

(1682, 19)

## Three ways to write a modality

### A numpy array and its row order

Published multimodal datasets ship each modality as a `.npy` matrix. A matrix
carries no identifiers, so WarpRec requires a second file, `item_path`, naming
the item of each row, one per line. The rows are written here in a shuffled
order, as a batched extraction job might write them: the row-order file is what
puts them back.

In [6]:
order = np.random.default_rng(42).permutation(len(items))
np.save(RUNS / "text.npy", text[order])
items.item_id.iloc[order].to_csv(RUNS / "text_items.tsv", index=False, header=False)
print(
    (RUNS / "text_items.tsv").read_text().splitlines()[:3],
    np.load(RUNS / "text.npy").shape,
)

['153', '380', '558'] (1682, 32)


### A tabular file

A tabular or parquet file carries the identifiers itself: **the first column
is the item**, whatever it is called, and every other column is a feature. No
row-order file is needed.

In [7]:
genre_table = genres.copy()
genre_table.insert(0, "item_id", items.item_id)
genre_table.to_csv(RUNS / "genres.tsv", sep="\t", index=False)
genre_table.iloc[:3, :6]

,item_id,Action,Adventure,Animation,Children's,Comedy
0,1,0.0,0.0,1.0,1.0,1.0
1,2,1.0,1.0,0.0,0.0,0.0
2,3,0.0,0.0,0.0,0.0,0.0


### A parquet file, with items missing

Feature dumps rarely cover the whole catalogue: a poster that failed to
download, a description that was never written. To see what WarpRec does about
it, this third modality is the same text vectors with about 10% of the films
left out, written as parquet (again with the item column first).

In [8]:
covered = np.random.default_rng(0).random(len(items)) > 0.1
partial = pd.DataFrame(text, columns=[f"t{i}" for i in range(text.shape[1])])
partial.insert(0, "item_id", items.item_id)
partial[covered].to_parquet(RUNS / "text_partial.parquet", index=False)
print(f"{covered.sum()} of {len(items)} films have a vector")

1511 of 1682 films have a vector


## Reading them from a configuration

`reader.multimodal` maps each modality's name, chosen freely, to how it is
read. `file_format` defaults to `numpy`, and `header` defaults to `false` for
every file of the section (the row-order file included), so the tabular file
with a header says so. `normalize: l2` scales every row to unit length.

In [9]:
print(Path("configs/multimodal.yml").read_text())

# Three modalities, one per file layout. Paths are relative to the guide's folder.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
  multimodal:
    text:                     # a .npy matrix and the item of each of its rows
      local_path: ../runs/07-multimodal-features/text.npy
      item_path: ../runs/07-multimodal-features/text_items.tsv
    genres:                   # a TSV whose first column is the item
      local_path: ../runs/07-multimodal-features/genres.tsv
      file_format: tabular
      header: true
    partial:                  # a parquet file covering ~90% of the films
      local_path: ../runs/07-multimodal-features/text_partial.parquet
      file_format: parquet
      normalize: l2
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
models:
  Pop: {}
evaluation:
  top_k: [10]
  metrics: [nDCG]



With the log on, reading reports each modality, then a table with each one's
width and how many items of the catalogue it covers.

In [10]:
from warprec.common.initialize import initialize_datasets
from warprec.data.reader import ReaderFactory
from warprec.utils.callback import WarpRecCallback
from warprec.utils.config.config import load_design_configuration

logger.enable("warprec")
config = load_design_configuration("configs/multimodal.yml")
dataset, _, _ = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
logger.disable("warprec")

08-10-2026 12:24:38 - WarpRec - 📝 Reading design configuration file in: configs/multimodal.yml


08-10-2026 12:24:38 - WarpRec - 📝 Reading process completed correctly.


08-10-2026 12:24:38 - WarpRec - 📝 Starting test splitting process with temporal_holdout splitting strategy.


08-10-2026 12:24:38 - WarpRec - 📝 Test splitting completed in : 0.01s


08-10-2026 12:24:38 - WarpRec - ✅ Splitting process over in 0.01s.


...

08-10-2026 12:24:38 - WarpRec - 📊 Number of users: 943      Number of items: 1648      Transactions: 89561


08-10-2026 12:24:38 - WarpRec - 📝 ------------------------------------------------------------------------


08-10-2026 12:24:38 - WarpRec - 📝 --------------------------------Test set--------------------------------


08-10-2026 12:24:38 - WarpRec - 📊 Number of users: 943      Number of items: 1335      Transactions: 10401


08-10-2026 12:24:38 - WarpRec - 📝 ------------------------------------------------------------------------


The catalogue is the training split, so coverage is counted against its 1648
items, not against the 1682 rows of a file: the films that only appear in the
test split are not in the catalogue, and their rows are not used.
The `partial` modality misses some catalogue items; they are **not dropped**.
A feature file with gaps would otherwise change the catalogue every model in
the run is compared on.

## The `MultiModalFeatures` object

The dataset holds the features as `dataset.multimodal`, already aligned with
the item indices. `dims` and `coverage` summarise it:

In [11]:
mm = dataset.multimodal
pd.DataFrame({"dims": mm.dims(), "coverage": mm.coverage()}).rename_axis("modality")

,dims,coverage
modality,,
text,32,1648
genres,19,1648
partial,32,1481


`get(name)` returns one modality as a tensor in item-index order, with one
extra row at the end: index `n_items` is the padding item (guide 1), and its
row is zero. `select(names)` returns several at once, every one by default.

In [12]:
n_items = dataset.info()["n_items"]
print(mm.names(), "| n_items", n_items, "| get('text')", tuple(mm.get("text").shape))
print(
    [tuple(t.shape) for t in mm.select()],
    [tuple(t.shape) for t in mm.select(["genres"])],
)

['text', 'genres', 'partial'] | n_items 1648 | get('text') (1649, 32)
[(1649, 32), (1649, 19), (1649, 32)] [(1649, 19)]


The shuffled rows have been put back in place: the row of Star Wars in the
dataset is the vector computed for Star Wars.

In [13]:
_, item_map = dataset.get_mappings()
star_wars = item_of["Star Wars (1977)"]
np.allclose(mm.get("text")[item_map[star_wars]].numpy(), text[row_of_item[star_wars]])

True

An item a modality does not cover gets a row of zeros, and with
`normalize: l2` every covered row has unit length while the zero rows and the
padding row stay zero:

In [14]:
norms = mm.get("partial").norm(dim=1)
missing = [item_map[i] for i in items.item_id[~covered] if i in item_map]
pd.Series(
    {
        "covered rows, min norm": norms[:n_items][norms[:n_items] > 0].min().item(),
        "covered rows, max norm": norms[:n_items].max().item(),
        "uncovered items": len(missing),
        "uncovered rows, max norm": norms[missing].max().item(),
        "padding row norm": norms[n_items].item(),
    }
)

covered rows, min norm        1.0
covered rows, max norm        1.0
uncovered items             167.0
uncovered rows, max norm      0.0
padding row norm              0.0
dtype: float64

Models are built from `info()`, which reports the width of every modality
under `modality_dims`:

In [15]:
dataset.info()["modality_dims"]

{'text': 32, 'genres': 19, 'partial': 32}

## Why the array needs its row order

Leaving `item_path` out of a `numpy` modality is a configuration error:

In [16]:
from pydantic import ValidationError

from warprec.utils.config import ModalityReading

try:
    ModalityReading(local_path=str(RUNS / "text.npy"))
except ValidationError as error:
    print(error.errors()[0]["msg"])

Value error, A feature array carries no item identifiers, so 'item_path' must name the file listing the item of each row.


The reason is that nothing could detect a wrong guess. Below, the same matrix
is paired with the order one would most likely assume, item ids ascending.
`MultiModalFeatures` can be built directly from arrays, an item mapping and a
per-modality normalisation, which is what the dataset does with the files. Both
versions have the same width and full coverage, and either would train without
a warning; only the second gives each film its own vector.

In [17]:
from warprec.data.entities import MultiModalFeatures

shuffled = np.load(RUNS / "text.npy")
assumed = MultiModalFeatures({"text": (shuffled, sorted(items.item_id))}, item_map)
named = MultiModalFeatures(
    {"text": (shuffled, pd.read_csv(RUNS / "text_items.tsv", header=None)[0].tolist())},
    item_map,
)
for label, features in [("assumed order", assumed), ("item_path", named)]:
    print(
        f"{label:14}",
        features.coverage(),
        nearest(features.get("text").numpy(), item_map, "Star Wars (1977)", 3),
    )

assumed order  {'text': 1648} ['Grand Day Out, A (1992)', 'Meet Me in St. Louis (1944)', 'Philadelphia Story, The (1940)']
item_path      {'text': 1648} ['Empire Strikes Back, The (1980)', 'Starship Troopers (1997)', 'Return of the Jedi (1983)']


## Choosing modalities per model

Every multimodal model takes a `modalities` parameter. Left out, the model reads
**every** configured modality, so adding one to the reader does not leave it
unused. Built through the registry with the dataset's features:

In [18]:
import warprec.recommenders  # noqa: F401  (registers the models)
from warprec.utils.registry import model_registry

vbpr_params = {
    "embedding_size": 32,
    "reg_weight": 1e-4,
    "batch_size": 1024,
    "epochs": 5,
    "learning_rate": 1e-3,
}
for modalities in [None, ["text", "genres"]]:
    model = model_registry.get(
        "VBPR",
        params={**vbpr_params, "modalities": modalities},
        info=dataset.info(),
        multimodal=dataset.multimodal,
        seed=42,
    )
    print(f"modalities={modalities}:", model.modality_names, model.modality_dims)
print([key for key in model.state_dict() if key.startswith("modality_")])

modalities=None: ['text', 'genres', 'partial'] [32, 19, 32]
modalities=['text', 'genres']: ['text', 'genres'] [32, 19]
['modality_text', 'modality_genres']


VBPR scores against the concatenation of the modalities it reads, so its
feature projection is as wide as their widths summed. The feature tables are
registered as buffers (the last line above), so a saved model carries them.

Asking for a modality the dataset does not carry is an error when the model is
built, and the message lists the ones it does carry:

In [19]:
try:
    model_registry.get(
        "VBPR",
        params={**vbpr_params, "modalities": ["visual"]},
        info=dataset.info(),
        multimodal=dataset.multimodal,
    )
except ValueError as error:
    print(error)

VBPR was asked for the modalities ['visual'], which the dataset does not carry. It carries ['text', 'genres', 'partial'].


The training pipeline (guide 10) catches the same mistake earlier, when it
loads the configuration, before any data is read. The design pipeline does not
validate its models at that point, so there it surfaces as the error above.

In [20]:
print(Path("configs/unknown-modality.yml").read_text())

# A training configuration (guide 10) whose model asks for a modality
# the reader does not configure.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
  multimodal:
    text:
      local_path: ../runs/07-multimodal-features/text.npy
      item_path: ../runs/07-multimodal-features/text_items.tsv
writer:
  dataset_name: ml-100k
  writing_method: local
  local_experiment_path: ../runs/07-multimodal-features/experiments
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
models:
  VBPR:
    embedding_size: 64
    modalities: [visual]
    reg_weight: 0.0001
    batch_size: 1024
    epochs: 5
    learning_rate: 0.001
evaluation:
  top_k: [10]
  metrics: [nDCG]



In [21]:
from warprec.utils.config.config import load_train_configuration

try:
    load_train_configuration("configs/unknown-modality.yml")
except ValueError as error:
    print(error.errors()[0]["msg"])

Value error, The model VBPR asks for the modalities ['visual'], which 'reader.multimodal' does not configure. The configured ones are ['text']. Check the configuration file.


## Training VBPR next to BPR

The design pipeline (guide 9) trains every model of a configuration once and
logs its test results. Here VBPR reads both modalities, since it names none;
BPR, with the same width and schedule, is the same model without the
features. Ten epochs on a CPU are enough to see the pipeline work, not to
compare the two models seriously.

In [22]:
print(Path("configs/train.yml").read_text())

# BPR and VBPR with the same width and schedule; VBPR also reads two modalities.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
  multimodal:
    text:
      local_path: ../runs/07-multimodal-features/text.npy
      item_path: ../runs/07-multimodal-features/text_items.tsv
    genres:
      local_path: ../runs/07-multimodal-features/genres.tsv
      file_format: tabular
      header: true
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
models:
  BPR:
    embedding_size: 64
    reg_weight: 0.0001
    batch_size: 1024
    epochs: 10
    learning_rate: 0.001
    optimization:
      num_workers: 0          # load batches in this process; no worker start-up
  VBPR:
    embedding_size: 64        # modalities left out: reads text and genres
    reg_weight: 0.0001
    batch_size: 1024
    epochs: 10
    learning_rate: 0.001
    optimization:
      num_workers: 0
eval

Lightning prints a summary of each model's layers. VBPR's `feature_projection`
maps the 32 + 19 = 51 feature columns onto the 64-wide embedding, and
`visual_bias` holds one weight per feature column; the feature tables
themselves are buffers, not parameters, so they are not trained.

In [23]:
import warnings

from warprec.pipelines.design import design_pipeline

logger.enable("warprec")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    design_pipeline("configs/train.yml")
logger.disable("warprec")

08-10-2026 12:24:38 - WarpRec - 📝 Starting the Design Pipeline.


08-10-2026 12:24:38 - WarpRec - 📝 Reading design configuration file in: configs/train.yml


08-10-2026 12:24:38 - WarpRec - 📝 Reading process completed correctly.


08-10-2026 12:24:38 - WarpRec - 📝 Starting test splitting process with temporal_holdout splitting strategy.


08-10-2026 12:24:38 - WarpRec - 📝 Test splitting completed in : 0.01s


...

08-10-2026 12:25:06 - WarpRec - 📝 | Top@10   | 0.12353  | 0.111897 |            338 |


08-10-2026 12:25:06 - WarpRec - 📝 +----------+----------+----------+----------------+


08-10-2026 12:25:06 - WarpRec - 📝 | Top@20   | 0.144079 | 0.183955 |            491 |


08-10-2026 12:25:06 - WarpRec - 📝 +----------+----------+----------+----------------+


08-10-2026 12:25:06 - WarpRec - ✅ Design pipeline executed successfully. WarpRec is shutting down.


After ten epochs VBPR reaches an nDCG@10 of 0.124 against BPR's 0.112, and its
top-10 lists cover 338 distinct items against 212. That is one short run on one
split, enough to show the features reach the model; guide 10 covers tuning and
guide 12 testing whether a difference is significant.

## Summary of the `multimodal` section

| key | what it does |
|---|---|
| `reader.multimodal` | a mapping of modality name → how to read it; any number of modalities |
| `local_path` | the feature file |
| `item_path` | the item of each row, one per line; required for `numpy` |
| `file_format` | `numpy` (default), `tabular` or `parquet`; the tabular formats carry the item in their first column |
| `sep`, `header` | for the tabular files and the row-order file; `header` defaults to `false` |
| `item_column_name` | the item column's name (default `item_id`), used to read a row-order file that has a header |
| `normalize` | `none` (default) or `l2`, per row; uncovered and padding rows stay zero |
| `models.<Model>.modalities` | which modalities a multimodal model reads; every one when omitted |

The reference pages are
[Reader configuration](https://warprec.readthedocs.io/en/latest/configuration/reader/)
(multimodal section) and
[Multimodal recommenders](https://warprec.readthedocs.io/en/latest/recommenders/multimodal/).